# 04 - Results and Evaluation
Evaluate model outputs and inspect arbitrage signals with explainability diagnostics.

In [3]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# Ensure project modules are importable when running from notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Ensure notebook kernel has required dependencies for this notebook.
for package in ["yfinance", "shap"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

from config import ARBITRAGE_WINDOW, PAIR_CONFIGS, VOLATILITY_FILTER_QUANTILE, ZSCORE_ENTRY, ZSCORE_EXIT
from src.arbitrage_detector import ArbitrageDetector
from src.data_loader import MarketDataLoader
from src.features import FeatureEngineer
from src.models import TrackingErrorModel

loader = MarketDataLoader()
panel = loader.fetch_universe(PAIR_CONFIGS, period='2y', interval='1d')
features = FeatureEngineer(rolling_window=20, horizon=1).transform_universe(panel)

model_path = project_root / "artifacts" / "te_model.joblib"
if not model_path.exists():
    raise FileNotFoundError(f"Model artifact not found at {model_path}. Run notebook 03 training save cell first.")

model = TrackingErrorModel.load(model_path)
detector = ArbitrageDetector(
    window=ARBITRAGE_WINDOW,
    zscore_entry=ZSCORE_ENTRY,
    zscore_exit=ZSCORE_EXIT,
    volatility_filter_quantile=VOLATILITY_FILTER_QUANTILE,
)

features_clean = features.dropna()
x = features_clean[model.numeric_columns + model.categorical_columns]
pred = model.predict(x)
pred[:5]

c:\Users\Alexandre\anaconda3\envs\catan\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


array([0.00037953, 0.00041809, 0.00068396, 0.00069541, 0.00042138])

In [4]:
pair_name = features_clean['pair'].iloc[-1]
pair_panel = panel[panel['pair'] == pair_name]
signal_snapshot = detector.latest_signal(pair_panel)
signal_snapshot

{'signal': 'WATCH',
 'spread_zscore': 1.415932490518419,
 'spread_vol': 0.00036312464806846607,
 'half_life': 71.51461447945236}

In [5]:
shap_df = model.explain_shap(x.tail(200), max_samples=100).head(10)
shap_df

,feature,mean_abs_shap
38,num__te_roll_std_20,0.001874
42,num__spread_std_20,0.000144
48,num__rolling_beta,0.000127
54,cat__pair_FEZ_^STOXX50E,0.000115
36,num__spread_mean_10,0.000088
33,num__te_roll_std_10,0.000072
39,num__etf_vol_20,0.000029
47,num__spread_std_60,0.000022
40,num__benchmark_vol_20,0.000020
32,num__spread_std_5,0.000019
